## **DO NOT rename or change the signature of these functions. Your code must be in the 3rd cell of the notebook, otherwise the tests will fall.**

# Homework: AI Agents

## Instructions
1. **"Template" cell** — run it first, do not modify.
2. **"Tasks" cell** — write your code where you see `# YOUR CODE HERE`.
3. Run the open examples and make sure all say `OK`.
4. Submit the notebook with saved outputs.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          TEMPLATE — DO NOT MODIFY THIS CELL                 ║
# ╚══════════════════════════════════════════════════════════════╝
# %pip install -q langchain-openai langchain-core

import os, json, copy
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

MODEL_NAME = "gpt-oss-20b"
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)


def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Sends the message history to the LLM and returns the model response.

    Parameters:
      messages — list of dialog messages. Each message is a LangChain object:
                   SystemMessage(content="...")   — instruction for the model (agent role)
                   HumanMessage(content="...")    — message from the user
                   AIMessage(...)                 — previous model response
                   ToolMessage(content="...", tool_call_id="...") — tool result

      tools   — list of tool descriptions (OpenAI function calling schema or LangChain tools).

    Returns AIMessage:
      msg.content    — text response (str)
      msg.tool_calls — list of tool calls:
                         "name" — tool name
                         "args" — arguments (already parsed dict)
                         "id"   — unique call identifier
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Product catalog
CATALOG = [
    {"id": "p1",  "name": "Sony WH-1000XM5",            "category": "headphones", "brand": "Sony",     "price": 349, "color": "black",    "rating": 4.8, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p2",  "name": "Sony WH-CH720N",              "category": "headphones", "brand": "Sony",     "price": 129, "color": "blue",     "rating": 4.4, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p3",  "name": "Bose QuietComfort Ultra",     "category": "headphones", "brand": "Bose",     "price": 379, "color": "white",    "rating": 4.7, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p4",  "name": "Apple AirPods Pro 2",         "category": "earbuds",    "brand": "Apple",    "price": 249, "color": "white",    "rating": 4.6, "tags": ["wireless", "noise-cancelling", "ios"]},
    {"id": "p5",  "name": "Anker Soundcore Liberty 4 NC","category": "earbuds",    "brand": "Anker",    "price": 99,  "color": "black",    "rating": 4.3, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p6",  "name": "Logitech MX Master 3S",       "category": "mouse",      "brand": "Logitech", "price": 109, "color": "graphite", "rating": 4.8, "tags": ["wireless", "productivity", "premium"]},
    {"id": "p7",  "name": "Logitech Pebble 2",           "category": "mouse",      "brand": "Logitech", "price": 34,  "color": "white",    "rating": 4.2, "tags": ["wireless", "budget", "portable"]},
    {"id": "p8",  "name": "Keychron K2",                 "category": "keyboard",   "brand": "Keychron", "price": 89,  "color": "black",    "rating": 4.5, "tags": ["wireless", "mechanical", "compact"]},
    {"id": "p9",  "name": "NuPhy Air75",                 "category": "keyboard",   "brand": "NuPhy",    "price": 139, "color": "gray",     "rating": 4.6, "tags": ["wireless", "mechanical", "low-profile"]},
    {"id": "p10", "name": "Amazon Kindle Paperwhite",    "category": "ereader",    "brand": "Amazon",   "price": 149, "color": "black",    "rating": 4.7, "tags": ["reading", "portable", "gift"]},
]


@dataclass
class ShopState:
    """Session state: cart and last search results."""
    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Collects all tool calls."""
    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Tool Call Trace ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:80]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Shop logic — search and add to cart."""
    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(self, query: str = "", category: str | None = None,
                        brand: str | None = None, max_price: float | None = None,
                        sort_by: str | None = None) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words): continue
            if category and item["category"] != category: continue
            if brand and item["brand"].lower() != brand.lower(): continue
            if max_price is not None and item["price"] > float(max_price): continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc": results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc": results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Product {product_id} not found"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append({"product_id": product_id, "name": product["name"],
                                "price": product["price"], "quantity": quantity})
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    """Shared context passed between agents in Task 3."""
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)   # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)
print("Template loaded.")
print(f"  Model: {MODEL_NAME}")
print(f"  Catalog: {len(CATALOG)} products")
print(f"  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool")
print(f"  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               YOUR CODE — THREE TASKS                        ║
# ╚══════════════════════════════════════════════════════════════╝

import os
import re

# ─── Yandex AI Studio override (local testing) ──────────────────
# Insert your Yandex AI Studio credentials below to run the notebook
# locally on the Yandex Jupyter server. If either value is empty, the
# original `llm` from the template cell is kept unchanged — that is what
# the grader uses.
_YANDEX_API_KEY = os.environ.get("YANDEX_API_KEY", "")  # AI Studio secret
_YANDEX_FOLDER_ID = os.environ.get("YANDEX_FOLDER_ID", "")  # Yandex Cloud folder id

if _YANDEX_API_KEY and _YANDEX_FOLDER_ID:
    os.environ["OPENAI_API_KEY"] = _YANDEX_API_KEY
    llm = ChatOpenAI(
        model=f"gpt://{_YANDEX_FOLDER_ID}/gpt-oss-20b",
        temperature=0,
        base_url="https://ai.api.cloud.yandex.net/v1",
        api_key=_YANDEX_API_KEY,
    )

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 1. Tool-Calling Agent (ReAct loop)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

_VALID_CATEGORIES = {"headphones", "earbuds", "keyboard", "mouse", "ereader"}
_VALID_SORTS = {"price_asc", "rating_desc"}


def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    """Search products in the store catalog. ALWAYS call this tool first when the user asks to find, look up, compare, recommend or choose a product.

    The tool returns a list of product dicts with keys: id, name, category, brand, price, color, rating, tags.

    Args:
        query: Free-text query. Every whitespace-separated word must appear (case-insensitive) in the product name, category, brand or tags. Pass "" to disable text matching and rely on structured filters instead. Do NOT put category names, brand names or price numbers here — use the dedicated parameters.
        category: Exact product category (lowercase). One of: "headphones", "earbuds", "keyboard", "mouse", "ereader". Pass null when the user does not specify a category.
        brand: Brand name, e.g. "Sony", "Apple", "Bose", "Logitech", "Keychron", "Anker", "Amazon", "NuPhy". Case-insensitive.
        max_price: Upper bound on price in US dollars (number, not string). Use this whenever the user mentions a budget, "under X", "below X", "cheaper than X", "at most X".
        sort_by: Sort order of results. Use "price_asc" when the user asks for the cheapest/lowest/least expensive item. Use "rating_desc" when the user asks for the best/top-rated/highest-rated item. Pass null otherwise.

    Returns:
        A list of matching product dicts. May be empty if nothing matches.
    """
    ...


def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    """Add a specific product to the user's shopping cart. Call this ONLY after search_products has returned the product and you know its exact id.

    Args:
        product_id: Catalog id of the product to add, e.g. "p1", "p6", "p10". This is the "id" field from a product returned by search_products.
        quantity: Number of units to add. Defaults to 1 when the user does not specify a quantity.

    Returns:
        Dict with fields: {"ok": bool, "cart_size": int} on success or {"ok": false, "error": str} on failure.
    """
    ...


SHOP_TOOLS_SCHEMA = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
]


def _coerce_str(value) -> str | None:
    if value is None:
        return None
    if isinstance(value, str):
        return value
    return str(value)


def _coerce_float(value) -> float | None:
    if value is None or value == "":
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _normalize_search_args(args: dict) -> dict:
    """Normalize LLM-provided search args: lowercase category/sort, coerce numerics, clean strings."""
    out: dict = {}

    query = args.get("query")
    if isinstance(query, list):
        query = " ".join(str(q) for q in query)
    out["query"] = (query or "").strip() if isinstance(query, str) else ""

    category = _coerce_str(args.get("category"))
    if category:
        category = category.strip().lower()
        if category.endswith("s") and category[:-1] in _VALID_CATEGORIES:
            category = category[:-1]
        if category not in _VALID_CATEGORIES:
            category = None
    out["category"] = category

    brand = _coerce_str(args.get("brand"))
    out["brand"] = brand.strip() if brand else None

    out["max_price"] = _coerce_float(args.get("max_price"))

    sort_by = _coerce_str(args.get("sort_by"))
    if sort_by:
        sort_by = sort_by.strip().lower().replace("-", "_")
        if sort_by in {"price", "asc", "cheapest", "lowest", "price_ascending"}:
            sort_by = "price_asc"
        elif sort_by in {"rating", "desc", "best", "top", "highest", "rating_descending"}:
            sort_by = "rating_desc"
        if sort_by not in _VALID_SORTS:
            sort_by = None
    out["sort_by"] = sort_by

    return out


def _execute_shop_tool_call(call: dict, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    """Dispatch a tool call to the ShopTools, update state and tracer, return JSON string for ToolMessage."""
    name = call.get("name", "") or ""
    args = call.get("args", {}) or {}

    if name == "search_products":
        normalized = _normalize_search_args(args)
        result = tools.search_products(**normalized)
        state.last_results = result
        tracer.record(name, args, result)
        return json.dumps(result, ensure_ascii=False)

    if name == "add_to_cart":
        product_id = _coerce_str(args.get("product_id"))
        quantity_raw = args.get("quantity", 1)
        try:
            quantity = int(quantity_raw) if quantity_raw is not None else 1
        except (TypeError, ValueError):
            quantity = 1
        if quantity < 1:
            quantity = 1
        result = tools.add_to_cart(state, product_id=product_id or "", quantity=quantity)
        tracer.record(name, args, result)
        return json.dumps(result, ensure_ascii=False)

    result = {"ok": False, "error": f"Unknown tool: {name}"}
    tracer.record(name, args, result)
    return json.dumps(result, ensure_ascii=False)


_SHOP_SYSTEM_PROMPT = (
    "You are a shop assistant for an online electronics store (headphones, "
    "earbuds, keyboards, mice, e-readers). You MUST use tools to answer any "
    "question about products or the cart. Do not answer from memory.\n"
    "Available tools:\n"
    "  - search_products(query, category, brand, max_price, sort_by) — find products.\n"
    "  - add_to_cart(product_id, quantity) — put a product into the cart.\n"
    "Rules:\n"
    "  1. ALWAYS call search_products first when the user asks to find, "
    "     recommend, compare or choose a product.\n"
    "  2. Extract filters from the user's message: category (headphones, "
    "     earbuds, keyboard, mouse, ereader — exactly these lowercase words), "
    "     brand (Sony, Apple, Bose, Logitech, Keychron, Anker, Amazon, NuPhy), "
    "     budget → max_price (number).\n"
    "  3. For 'cheapest/lowest price' use sort_by='price_asc'. For 'best/top "
    "     rating' use sort_by='rating_desc'.\n"
    "  4. If the user asks to add/buy/put into cart, call add_to_cart AFTER "
    "     you have the product id from search_products. Choose the correct "
    "     product id based on the user's intent (cheapest, best rated, etc.) "
    "     from the search results.\n"
    "  5. When all tools are done, give a short final text reply in plain "
    "     English (no JSON, no tool calls)."
)


def run_shopping_agent(user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    """ReAct shopping agent: iteratively calls the LLM with SHOP_TOOLS_SCHEMA, executes returned tool calls, returns final text."""
    messages: list = [
        SystemMessage(content=_SHOP_SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ]

    max_iterations = 10
    last_text = ""

    for _ in range(max_iterations):
        ai_msg = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA)
        messages.append(ai_msg)

        tool_calls = getattr(ai_msg, "tool_calls", None) or []
        if not tool_calls:
            last_text = ai_msg.content or ""
            return last_text

        for call in tool_calls:
            content = _execute_shop_tool_call(call, state, tools, tracer)
            messages.append(ToolMessage(content=content, tool_call_id=call.get("id", "") or ""))

    final = messages[-1] if messages else None
    return getattr(final, "content", "") or last_text


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 2. Memory Agent
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PROFILE_PATH = Path("user_profile.json")


def load_profile(path: Path = PROFILE_PATH) -> dict:
    """Loads profile from JSON. Returns {} if file does not exist."""
    path = Path(path)
    if not path.exists():
        return {}
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except (json.JSONDecodeError, OSError):
        return {}


def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    """Saves the profile dict to a file as JSON."""
    path = Path(path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(profile, f, ensure_ascii=False, indent=2)


def update_profile(key: str, value: str) -> dict:
    """Persist one user preference to the long-term profile. Call this the moment the user reveals a personal preference you want to remember across sessions.

    Args:
        key: The preference slot. Use exactly one of: "name" (the user's personal name), "brand" (preferred brand), "max_price" (budget in dollars as a string), "color" (preferred color), "category" (category of interest: headphones, earbuds, keyboard, mouse, ereader).
        value: The new value as a string. For max_price pass a numeric string like "200".

    Returns:
        Dict: {"ok": true, "profile": <updated profile dict>}.
    """
    ...


SHOP_TOOLS_SCHEMA_WITH_MEMORY = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
    convert_to_openai_tool(update_profile),
]


def _execute_memory_tool_call(
    call: dict,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    profile: dict,
    profile_path: Path,
) -> str:
    """Dispatch a tool call including update_profile, persist profile changes to disk."""
    name = call.get("name", "") or ""
    args = call.get("args", {}) or {}

    if name == "update_profile":
        key = _coerce_str(args.get("key"))
        value = args.get("value")
        if isinstance(value, (dict, list)):
            value = json.dumps(value, ensure_ascii=False)
        elif value is not None:
            value = str(value)
        if key:
            profile[key] = value
            save_profile(profile, profile_path)
        result = {"ok": True, "profile": profile}
        tracer.record(name, args, result)
        return json.dumps(result, ensure_ascii=False)

    return _execute_shop_tool_call(call, state, tools, tracer)


def _build_memory_system_prompt(profile: dict) -> str:
    profile_block = json.dumps(profile, ensure_ascii=False) if profile else "{}"
    return (
        "You are a shop assistant with MEMORY for an electronics store "
        "(headphones, earbuds, keyboards, mice, e-readers).\n\n"
        f"User profile (persisted between sessions): {profile_block}\n\n"
        "Available tools:\n"
        "  - search_products(query, category, brand, max_price, sort_by) — find products.\n"
        "  - add_to_cart(product_id, quantity) — put a product into the cart.\n"
        "  - update_profile(key, value) — persist a user preference.\n\n"
        "Rules:\n"
        "  1. When the user reveals a personal fact (name, preferred brand, "
        "     budget/max_price, color, category), IMMEDIATELY call "
        "     update_profile — one call per slot. Do NOT wait. Then confirm "
        "     briefly in plain text.\n"
        "  2. When the user asks about themselves ('what is my name?', 'what "
        "     is my budget?') answer directly from the profile above, without "
        "     calling any tools.\n"
        "  3. When the user asks to find/recommend a product, call "
        "     search_products. Apply the profile preferences as defaults "
        "     (e.g. if profile.brand is set and the user did not override "
        "     it, filter by that brand; if profile.max_price is set, use it "
        "     as the budget).\n"
        "  4. When the user refers to the previous results ('the first one', "
        "     'that one', 'the cheapest you showed'), use the search results "
        "     already visible in the conversation history to pick a "
        "     product_id — do NOT search again.\n"
        "  5. For add/buy/put-into-cart requests, call add_to_cart with the "
        "     correct product_id.\n"
        "  6. Categories are lowercase: headphones, earbuds, keyboard, "
        "     mouse, ereader. For 'cheapest' use sort_by='price_asc'; for "
        "     'best rated' use sort_by='rating_desc'.\n"
        "  7. Finish every turn with a concise plain-text reply."
    )


def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple:
    """Memory-enabled ReAct agent. Combines a persisted profile (long-term) with chat history (short-term)."""
    profile = load_profile(profile_path)

    new_history = list(history)
    new_history.append(HumanMessage(content=user_message))

    def _build_messages() -> list:
        # Rebuild the system prompt each time so freshly saved profile entries are visible.
        return [SystemMessage(content=_build_memory_system_prompt(profile))] + list(new_history)

    final_text = ""
    max_iterations = 10

    for _ in range(max_iterations):
        messages = _build_messages()
        ai_msg = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA_WITH_MEMORY)
        new_history.append(ai_msg)

        tool_calls = getattr(ai_msg, "tool_calls", None) or []
        if not tool_calls:
            final_text = ai_msg.content or ""
            break

        for call in tool_calls:
            content = _execute_memory_tool_call(call, state, tools, tracer, profile, profile_path)
            new_history.append(ToolMessage(content=content, tool_call_id=call.get("id", "") or ""))
    else:
        for msg in reversed(new_history):
            text = getattr(msg, "content", "") or ""
            if text:
                final_text = text
                break

    return final_text, new_history


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 3. Multi-Agent System
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext


def _extract_max_price_from_text(text: str) -> float | None:
    """Heuristic fallback: parse 'under 120 dollars' / 'below $150' / 'up to 100'."""
    if not text:
        return None
    patterns = [
        r"under\s+\$?(\d+(?:\.\d+)?)",
        r"below\s+\$?(\d+(?:\.\d+)?)",
        r"less than\s+\$?(\d+(?:\.\d+)?)",
        r"cheaper than\s+\$?(\d+(?:\.\d+)?)",
        r"up to\s+\$?(\d+(?:\.\d+)?)",
        r"no more than\s+\$?(\d+(?:\.\d+)?)",
        r"budget.*?\$?(\d+(?:\.\d+)?)",
        r"max(?:imum)?\s+\$?(\d+(?:\.\d+)?)",
        r"<=\s*\$?(\d+(?:\.\d+)?)",
        r"<\s*\$?(\d+(?:\.\d+)?)",
    ]
    lowered = text.lower()
    for pat in patterns:
        m = re.search(pat, lowered)
        if m:
            try:
                return float(m.group(1))
            except ValueError:
                continue
    return None


class RetrieverAgent:
    """LLM with only the search tool. Fills ctx.candidates (up to 5) and ctx.max_price."""

    _SYSTEM = (
        "You are a product retrieval specialist for an electronics store. "
        "Your ONLY job: call search_products with the best filters you can "
        "derive from the user's query, then stop. You must not add anything "
        "to the cart, analyse pros/cons, or rank items — other agents will "
        "do that.\n"
        "Categories (lowercase): headphones, earbuds, keyboard, mouse, ereader.\n"
        "Map 'cheapest' -> sort_by='price_asc'; 'best/top-rated' -> "
        "sort_by='rating_desc'. Extract budget -> max_price (number). "
        "After you see relevant results, reply with a short confirmation "
        "like 'Candidates found.'"
    )

    def run(self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentContext:
        search_only_schema = [convert_to_openai_tool(search_products)]

        messages: list = [
            SystemMessage(content=self._SYSTEM),
            HumanMessage(content=ctx.query),
        ]

        collected: list[dict] = []
        max_iterations = 4

        for _ in range(max_iterations):
            ai_msg = llm_chat(messages, tools=search_only_schema)
            messages.append(ai_msg)

            tool_calls = getattr(ai_msg, "tool_calls", None) or []
            if not tool_calls:
                break

            for call in tool_calls:
                name = call.get("name", "") or ""
                args = call.get("args", {}) or {}

                if name != "search_products":
                    err = {"ok": False, "error": f"Tool {name} is not allowed here."}
                    tracer.record(name, args, err)
                    messages.append(ToolMessage(
                        content=json.dumps(err, ensure_ascii=False),
                        tool_call_id=call.get("id", "") or "",
                    ))
                    continue

                normalized = _normalize_search_args(args)
                result = tools.search_products(**normalized)
                state.last_results = result
                tracer.record(name, args, result)

                if normalized.get("max_price") is not None:
                    ctx.max_price = normalized["max_price"]

                collected.extend(result)
                messages.append(ToolMessage(
                    content=json.dumps(result, ensure_ascii=False),
                    tool_call_id=call.get("id", "") or "",
                ))

        if ctx.max_price is None:
            fallback = _extract_max_price_from_text(ctx.query)
            if fallback is not None:
                ctx.max_price = fallback

        seen: set[str] = set()
        unique: list[dict] = []
        for p in collected:
            pid = p.get("id")
            if pid and pid not in seen:
                seen.add(pid)
                unique.append(p)
            if len(unique) >= 5:
                break

        ctx.candidates = unique
        return ctx


class ProsAgent:
    """LLM without tools. Writes 1-2 sentences of pros per candidate, stored in ctx.pros."""

    _SYSTEM = (
        "You are a product analyst. For the given product write 1-2 concise "
        "sentences describing its main pros and strengths (value, features, "
        "target audience). Plain text only."
    )

    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        pros: dict[str, str] = {}
        for product in ctx.candidates:
            messages = [
                SystemMessage(content=self._SYSTEM),
                HumanMessage(content=(
                    "Write the pros for this product:\n"
                    f"{json.dumps(product, ensure_ascii=False)}"
                )),
            ]
            ai_msg = llm_chat(messages)
            pros[product["id"]] = (ai_msg.content or "").strip()

        ctx.pros = pros
        tracer.record("analyze_pros", {"count": len(ctx.candidates)}, {"ids": list(pros.keys())})
        return ctx


class ConsAgent:
    """LLM without tools. Writes 1-2 sentences of cons per candidate, stored in ctx.cons."""

    _SYSTEM = (
        "You are an honest product reviewer. For the given product write 1-2 "
        "concise sentences describing its main cons, weaknesses or trade-offs "
        "(price, limitations, missing features). Plain text only."
    )

    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        cons: dict[str, str] = {}
        for product in ctx.candidates:
            messages = [
                SystemMessage(content=self._SYSTEM),
                HumanMessage(content=(
                    "Write the cons for this product:\n"
                    f"{json.dumps(product, ensure_ascii=False)}"
                )),
            ]
            ai_msg = llm_chat(messages)
            cons[product["id"]] = (ai_msg.content or "").strip()

        ctx.cons = cons
        tracer.record("analyze_cons", {"count": len(ctx.candidates)}, {"ids": list(cons.keys())})
        return ctx


class RankerAgent:
    """Deterministic ranker: filter by max_price, then sort by (-rating, price) and take the top."""

    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        pool = list(ctx.candidates)

        if ctx.max_price is not None:
            try:
                cap = float(ctx.max_price)
                pool = [p for p in pool if float(p.get("price", 0)) <= cap]
            except (TypeError, ValueError):
                pass

        if pool:
            pool.sort(key=lambda p: (-float(p.get("rating", 0)), float(p.get("price", 0))))
            ctx.best = pool[0]
        else:
            ctx.best = None

        tracer.record(
            "rank_candidates",
            {"count": len(ctx.candidates), "max_price": ctx.max_price},
            {"best_id": ctx.best["id"] if ctx.best else None},
        )
        return ctx


def _wants_cart(message: str) -> bool:
    """Detect whether the user asked to add the chosen item to the cart."""
    if not message:
        return False
    lowered = message.lower()
    phrases = [
        "add to cart", "add it to cart", "add to my cart", "add to the cart",
        "put in cart", "put it in cart", "put into cart", "into the cart",
        "buy it", "purchase it", "order it", "check out",
    ]
    if any(p in lowered for p in phrases):
        return True
    return "cart" in lowered and ("add" in lowered or "put" in lowered)


class CoordinatorAgent:
    """Orchestrates Retriever -> Pros -> Cons -> Ranker (-> Cart). Returns AgentResult."""

    def __init__(self):
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        self.ranker = RankerAgent()

    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        tracer = ToolTracer()
        ctx = AgentContext(query=user_message)
        trace: list = []

        trace.append("delegate_retriever")
        ctx = self.retriever.run(ctx, state, tools, tracer)

        trace.append("delegate_pros")
        ctx = self.pros_agent.run(ctx, tracer)

        trace.append("delegate_cons")
        ctx = self.cons_agent.run(ctx, tracer)

        trace.append("delegate_ranker")
        ctx = self.ranker.run(ctx, tracer)

        cart_line = ""
        if _wants_cart(user_message) and ctx.best is not None:
            trace.append("delegate_cart")
            cart_result = tools.add_to_cart(state, product_id=ctx.best["id"], quantity=1)
            ctx.cart_result = cart_result
            if cart_result.get("ok"):
                cart_line = f"\nAdded '{ctx.best['name']}' to your cart."
            else:
                cart_line = f"\nCould not add to cart: {cart_result.get('error', 'unknown error')}."

        if ctx.best is None:
            tail = f" under ${ctx.max_price:.0f}" if ctx.max_price is not None else ""
            response = f"Sorry, I could not find any product matching your request{tail}."
            return AgentResult(response=response, trace=trace, context=ctx)

        best = ctx.best
        pros_text = (ctx.pros.get(best["id"], "") or "").strip() or "—"
        cons_text = (ctx.cons.get(best["id"], "") or "").strip() or "—"

        response = (
            f"Best recommendation: {best['name']} — ${best['price']}, "
            f"rating {best['rating']}.\n"
            f"Pros: {pros_text}\n"
            f"Cons: {cons_text}"
            f"{cart_line}"
        )

        return AgentResult(response=response, trace=trace, context=ctx)


In [ ]:
# --- Open examples for Task 1 -------------------------------------------

# [1.A] Search with price filter
_s1a = ShopState(); _t1a = ToolTracer()
_r1a = run_shopping_agent("Find wireless headphones under 150 dollars", _s1a, TOOLS, _t1a)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products was not called"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")

# [1.B] Search + add the cheapest
_s1b = ShopState(); _t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Find a wireless mouse under 120 dollars and add the cheapest one to cart",
    _s1b, TOOLS, _t1b
)
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")

# [1.C] Best keyboard
_s1c = ShopState(); _t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Find a wireless keyboard with the best rating and add it to cart",
    _s1c, TOOLS, _t1c
)
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (rating {added['rating']})")


In [ ]:
# --- Open examples for Task 2 -------------------------------------------

# [2.A] Saving preferences
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists(): _p2a.unlink()
_s2a = ShopState(); _t2a = ToolTracer(); _h2a = []
_r2a, _h2a = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s2a, TOOLS, _t2a, _h2a, _p2a
)
_prof2a = load_profile(_p2a)
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] New session uses profile (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Boris", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState(); _t2b = ToolTracer(); _h2b = []
_r2b, _ = run_memory_agent("What is my name and what is my budget?", _s2b, TOOLS, _t2b, _h2b, _p2b)
assert "Boris" in _r2b
print("OK 2.B")

# [2.C] Short-term memory — turn 2 remembers turn 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists(): _p2c.unlink()
_s2c = ShopState(); _h2c = []
_, _h2c = run_memory_agent(
    "Find wireless headphones under 150 dollars", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Add the first one found to cart", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: added '{_s2c.cart[0]['name']}'")


In [ ]:
# --- Open examples for Task 3 -------------------------------------------

# [3.A] Full cycle: search -> pros -> cons -> ranking -> cart
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Search only, no add to cart
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Find a wireless keyboard", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")

# [3.C] RankerAgent — price tie-break with equal rating
_ctx3c = AgentContext(query="test", candidates=[
    {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
    {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
    {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
])
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")

# [3.D] RankerAgent respects ctx.max_price
_ctx3d = AgentContext(
    query="mouse under 120 dollars",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Super Mouse",  "price": 200, "rating": 4.9},
        {"id": "p6",        "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7",        "name": "Pebble 2",      "price": 34,  "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: context passed correctly, max_price is respected")
